# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 3 we turned each card's history into a flat token stream. Here we pretrain the foundation model — a Llama causal decoder — on those sequences by next-token prediction, using Ray Train to run a plain-PyTorch training loop across the cluster.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd
import logging
import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Pretrain by predicting the next token

We pretrain the model the way a language model learns text: **next-token prediction** over each card's flat token stream. Given the history so far, the decoder predicts the next token; every position contributes a gradient, so one packed sequence yields hundreds of self-supervised predictions. No labels are involved — the transactions supervise themselves, which is what lets a foundation model learn from far more data than the scarce fraud labels alone.

The model is a **Llama causal decoder** (`src/model.py`, built from `transformers` — GQA, SwiGLU, RMSNorm, RoPE), sized per scale (`full` ≈ 29M params, matching NVIDIA's blueprint). "Causal" means each position attends only to earlier ones, so predicting the next token can't cheat by peeking ahead — and the final position's hidden state, which has seen the whole history, becomes the customer embedding in Part 5.

## The training loop is plain PyTorch — Ray Train scales it

The function that does the work (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, forward, backward, step — with an AdamW optimizer on a **warmup + cosine-decay** learning-rate schedule (schedule, betas, and weight-decay come from the scale config, matching NVIDIA's recipe: lr 2e-4, β₂ 0.95, weight-decay 0.077). Ray Train wraps it with the distributed machinery you'd otherwise hand-write — it launches the workers, shards the dataset across them (`get_dataset_shard` + `iter_torch_batches`), wraps the model in DDP (`prepare_model`), and persists checkpoints to shared storage.

The payoff is the `ScalingConfig`: moving from one CPU worker at `mini` to 8 GPU workers at `full` is a one-line change (`num_workers`, `use_gpu`), not a rewrite of the loop. `scripts/03_pretrain.py` wraps this same `train_func` for headless runs, so the walkthrough and the job train identically.

Note: `mini` is a fast CPU smoke test. At `full` (8 epochs, 8×A10G GPU workers, 4096-token sequences) budget a couple of hours; a full run reaches perplexity ~1.7.

In [ ]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # train_func is shared with scripts/03_pretrain.py
import numpy as np
import math

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]
vocab_path = os.path.join(paths["nvcorpus"], "vocab.json")

# The corpus from Part 3 is ids.npy / attn.npy (NVIDIA-tokenizer output). Load it as a Ray
# Dataset of (input_ids, attention_mask) rows — the same columns train_func expects.
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"))
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"))
train_ds = (ray.data.from_numpy(ids).rename_columns({"data": "input_ids"})
            .zip(ray.data.from_numpy(attn).rename_columns({"data": "attention_mask"})))
n_seqs = int(ids.shape[0])

# The cosine LR schedule steps once per optimizer step, so it needs the total step count
# up front: (sequences / workers / batch) * epochs.
steps_per_epoch = max(1, math.ceil(n_seqs / pt["num_workers"] / pt["batch_size"]))
total_steps = steps_per_epoch * pt["epochs"]
warmup_steps = int(pt.get("warmup_ratio", 0.0) * total_steps)
print(f"pretrain (causal LM): {n_seqs:,} sequences · global batch {pt['batch_size'] * pt['num_workers']} "
      f"· ~{steps_per_epoch} steps/epoch × {pt['epochs']} = ~{total_steps:,} optimizer steps "
      f"(warmup {warmup_steps})")

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": vocab_path, "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        # AdamW betas/weight-decay + warmup/cosine LR schedule (from the scale config).
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    # The one line that moves laptop -> cluster: number of workers, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name=f"transaction_fm_pretrain_{SCALE}",      # scale-specific → no cross-scale resume clash
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy weights to the canonical path for Part 5
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

## Export the decoder for embedding

Part 5 embeds transactions by running them through this decoder with NVIDIA's `HuggingFaceDecoderInference`, which loads a model via `AutoModelForCausalLM.from_pretrained`. So we save our trained checkpoint as a HuggingFace directory (the inner `LlamaForCausalLM`).

In [ ]:
import torch
from src.model import build_model

# Export the trained decoder as a HuggingFace directory so Part 5's embedder can load it
# (NVIDIA's HuggingFaceDecoderInference calls AutoModelForCausalLM.from_pretrained). We load our
# TransactionFM checkpoint and save its inner LlamaForCausalLM (m.lm) in HF format.
ck = paths["checkpoint"]
with open(os.path.join(ck, "model_config.json")) as f:
    mc = json.load(f)
m = build_model(vocab_path=os.path.join(ck, "vocab.json"), arch=mc["arch"], max_len=mc["max_len"])
sd = torch.load(os.path.join(ck, "model.pt"), map_location="cpu")
sd = sd["model_state_dict"] if isinstance(sd, dict) and "model_state_dict" in sd else sd
missing, unexpected = m.load_state_dict(sd, strict=False)
os.makedirs(paths["hf"], exist_ok=True)
m.lm.save_pretrained(paths["hf"])
print(f"exported HF decoder -> {paths['hf']}  (state_dict: missing {len(missing)}, unexpected {len(unexpected)})")

## Reading the metrics

Watch **causal-LM loss** (cross-entropy) and its exponent, **perplexity** — the model's effective "how many tokens am I choosing between." Perplexity well below the vocabulary size (6,251) means the decoder has learned real next-token structure rather than guessing uniformly; it should fall epoch over epoch and settle as the cosine LR decays.

At `mini` (a couple of CPU epochs on a tiny model, few sequences) this is only a smoke test — the number that matters is the downstream fraud lift in Part 6, from the `full` GPU pretrain (perplexity ~1.7).

In [ ]:
m = result.metrics
with open(os.path.join(paths["nvcorpus"], "vocab.json")) as f:
    vocab_size = json.load(f)["vocab_size"]
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")
print(f"vocab {vocab_size:,} tokens — perplexity well below vocab size means the decoder "
      "learned real next-token structure (not uniform guessing).")

## Takeaways

**Ray Train**
- The training loop is plain PyTorch; Ray Train adds the distributed parts — worker launch, dataset sharding, DDP/FSDP, checkpointing, fault tolerance — without changing the loop.
- `ScalingConfig` is the one knob that moves the same code from one CPU worker at `mini` to 8 GPU workers at `full` (`num_workers`, `use_gpu`); the model fits one GPU at every scale (data-parallel DDP), with `use_fsdp` available if it ever outgrows one.
- Checkpoints persist to shared storage (`storage_path`), so workers on other nodes can write them and downstream stages can read them.
- The notebook runs the same `train_func` that `scripts/03_pretrain.py` runs headless.

**The model**
- Self-supervised **next-token prediction** over the NVIDIA-tokenized stream — a Llama causal decoder, no labels required.
- Built from `transformers` (`LlamaConfig`/`LlamaForCausalLM`) at NVIDIA's blueprint config, so the architecture is faithful, and pretrained with NVIDIA's optimizer recipe.
- Exported to a HuggingFace directory so Part 5's embedder can load it; the trained decoder's last-token hidden state becomes the customer embedding.

---

## Next

**Part 5 — Batch embedding extraction**: run the pretrained decoder over each transaction with Ray — a CPU-read + GPU-infer pipeline that writes one embedding per transaction.